
# Cross-Sectional Regression Configurations (A–H)
This notebook constructs a synthetic 5-year monthly panel of 500 equities across 10 sectors and implements the eight cross-sectional regression specifications summarised in the table below. The goal is to reproduce the expected relationships between the designs (e.g., Frisch–Waugh–Lovell equivalences, intercept interpretations, scaling behaviour) using explicit date- and sector-aware transformations.


| ID | Target (y) | Factors (X) | Sector dummies? | Intercept (alpha) | Dummies (delta_s) | β meaning (units) | tstats | Equal to (FWL) | When to use | Caveats |
| -- | ------------ | ------------ | ---------------- | ------------------ | ------------------ | ------------------ | ------ | --------------- | ----------- | ------- |
| A  | Raw (r) | Raw (x) | No | mean(r) − βᵀ mean(x) | – | Mixes within + between sector | Not sector‑neutral; t differ vs B/C/D | – | Almost never (bias risk) | Sector premia bias β if sector means of r and x correlate |
| B  | Raw (r) | Raw (x) | Yes | r̄0 − βᵀ x̄0 | (r̄s − r̄0) − βᵀ (x̄s − x̄0) | Within‑sector, raw units | Same as C/D | ≡ D | Baseline for sector premia + raw‑unit β | α depends on base‑sector means |
| C  | Raw (r) | x̃ (demean within s) | Yes | r̄0 | r̄s − r̄0 | Within‑sector, raw units | Same as B/D | ≡ D (same β, SEs) | Cleaner α/Δ bookkeeping vs B | Predictions identical to B |
| D  | r̃ (demean within s) | x̃ | No | ≈ 0 | – | Within‑sector, raw units | Same as B/C | ≡ B/C | Lean design; same β/SE/fits in demeaned space | Must add back sector means for raw‑unit predictions |
| E  | Raw (r) | x^z (z in s) | Yes | r̄0 | r̄s − r̄0 | Per 1 within‑s SD of x, change in raw r | t change vs B/C/D | – | Stable β scale across dates | Sector‑wise rescaling reweights sectors |
| F  | r^z (z in s) | x^z | No | 0 (by std) | – | Within‑s SD units for both sides (correlation‑like) | t change vs B/C/D | – | Pure variance‑balanced within‑s signal | Back‑transform for raw‑unit predictions |
| G  | Raw (r) | x̃ | No | mean(r) | – | Within‑sector, raw units | t differ vs B/C/D; intercept often large | ≠ B (unless r also demeaned) | Quick “partial‑out X only” | Leaves sector premia in r; α large; SEs awkward |
| H  | Global z: r*=(r−r̄)/s_r | Global z: x*=(x−x̄)/s_x | Yes/No | Depends on coding | Depends | β rescaled only | Invariant; same as A | – | Standardize for numerics/comparability | Not sector‑neutral |


In [1]:

import random
import math
from datetime import date, timedelta
from collections import OrderedDict

random.seed(42)

N_STOCKS = 500
N_MONTHS = 60
N_FACTORS = 5
FACTOR_NAMES = [f"x{i}" for i in range(1, N_FACTORS + 1)]
SECTORS = [f"Sector_{i}" for i in range(10)]
BASE_SECTOR = SECTORS[0]


def month_end(year: int, month: int) -> date:
    if month == 12:
        next_year, next_month = year + 1, 1
    else:
        next_year, next_month = year, month + 1
    first_next = date(next_year, next_month, 1)
    return first_next - timedelta(days=1)


def build_dates():
    dates = []
    year, month = 2018, 1
    for _ in range(N_MONTHS):
        dates.append(month_end(year, month))
        month += 1
        if month == 13:
            month = 1
            year += 1
    return dates


def finalize_scalar(stat):
    count = stat["count"]
    if count == 0:
        return {"mean": 0.0, "std": 0.0, "count": 0}
    mean = stat["sum"] / count
    variance = (stat["sumsq"] / count) - mean * mean
    if variance < 0:
        variance = 0.0
    std = math.sqrt(variance)
    return {"mean": mean, "std": std, "count": count}


def simulate_data():
    dates = build_dates()
    stocks = []
    for i in range(N_STOCKS):
        stocks.append({
            "id": f"Stock{i:04d}",
            "sector": random.choice(SECTORS),
            "base_loadings": [random.gauss(0.0, 0.7) for _ in range(N_FACTORS)],
        })

    data_by_date = OrderedDict()
    for current_date in dates:
        records = []
        factor_premia = [base + random.gauss(0.0, 0.01) for base in (0.04, -0.03, 0.02, 0.0, 0.0)]
        sector_shocks = {sec: random.gauss(0.0, 0.015) for sec in SECTORS}
        base_return = 0.005 + random.gauss(0.0, 0.005)
        for stock in stocks:
            exposures = [stock["base_loadings"][idx] + random.gauss(0.0, 0.2) for idx in range(N_FACTORS)]
            sensitivity = sum(exposures[f_idx] * factor_premia[f_idx] for f_idx in range(N_FACTORS))
            residual = random.gauss(0.0, 0.02)
            realised = base_return + sensitivity + sector_shocks[stock["sector"]] + residual
            record = {
                "date": current_date,
                "stock_id": stock["id"],
                "sector": stock["sector"],
                "return": realised,
            }
            for idx, fname in enumerate(FACTOR_NAMES):
                record[fname] = exposures[idx]
            records.append(record)
        data_by_date[current_date] = records
    return data_by_date


def compute_date_stats(data_by_date):
    stats = {}
    for current_date, records in data_by_date.items():
        overall_return_stat = {"sum": 0.0, "sumsq": 0.0, "count": 0}
        overall_factor_stats = {
            fname: {"sum": 0.0, "sumsq": 0.0, "count": 0} for fname in FACTOR_NAMES
        }
        sector_stats = {
            sec: {
                "return": {"sum": 0.0, "sumsq": 0.0, "count": 0},
                "factors": {fname: {"sum": 0.0, "sumsq": 0.0, "count": 0} for fname in FACTOR_NAMES},
            }
            for sec in SECTORS
        }
        for rec in records:
            ret = rec["return"]
            overall_return_stat["sum"] += ret
            overall_return_stat["sumsq"] += ret * ret
            overall_return_stat["count"] += 1
            sec = rec["sector"]
            sec_stat = sector_stats[sec]
            sec_stat["return"]["sum"] += ret
            sec_stat["return"]["sumsq"] += ret * ret
            sec_stat["return"]["count"] += 1
            for fname in FACTOR_NAMES:
                val = rec[fname]
                overall_factor_stats[fname]["sum"] += val
                overall_factor_stats[fname]["sumsq"] += val * val
                overall_factor_stats[fname]["count"] += 1
                fstat = sec_stat["factors"][fname]
                fstat["sum"] += val
                fstat["sumsq"] += val * val
                fstat["count"] += 1
        overall = {
            "return": finalize_scalar(overall_return_stat),
            "factors": {fname: finalize_scalar(overall_factor_stats[fname]) for fname in FACTOR_NAMES},
        }
        sectors = {
            sec: {
                "return": finalize_scalar(sec_stats["return"]),
                "factors": {fname: finalize_scalar(sec_stats["factors"][fname]) for fname in FACTOR_NAMES},
            }
            for sec, sec_stats in sector_stats.items()
        }
        stats[current_date] = {"overall": overall, "sectors": sectors}
    return stats


In [2]:

data_by_date = simulate_data()
date_stats = compute_date_stats(data_by_date)

first_date = next(iter(data_by_date))
print(f"First month: {first_date} | n = {len(data_by_date[first_date])}")
print("Sample rows:")
for row in data_by_date[first_date][:3]:
    print(row)

sector_counts = {sec: date_stats[first_date]['sectors'][sec]['return']['count'] for sec in SECTORS}
print("Sector counts on first month:", sector_counts)


First month: 2018-01-31 | n = 500
Sample rows:
{'date': datetime.date(2018, 1, 31), 'stock_id': 'Stock0000', 'sector': 'Sector_1', 'return': 0.029855309014004332, 'x1': 0.5795895552951686, 'x2': 0.13752999967943152, 'x3': 0.4461557246201986, 'x4': 1.220841037412513, 'x5': -0.43037439668777033}
{'date': datetime.date(2018, 1, 31), 'stock_id': 'Stock0001', 'sector': 'Sector_1', 'return': -0.05232707720761853, 'x1': -1.4590654867327524, 'x2': 0.0904856600491642, 'x3': -0.03239223137504596, 'x4': 0.2524701979690631, 'x5': 0.4414238945136403}
{'date': datetime.date(2018, 1, 31), 'stock_id': 'Stock0002', 'sector': 'Sector_9', 'return': 0.04142331022916413, 'x1': 0.5070483752093833, 'x2': 0.025729222158435733, 'x3': -0.3295999259244433, 'x4': -0.4602100344947023, 'x5': 0.23347888389292454}
Sector counts on first month: {'Sector_0': 39, 'Sector_1': 64, 'Sector_2': 57, 'Sector_3': 47, 'Sector_4': 46, 'Sector_5': 53, 'Sector_6': 53, 'Sector_7': 45, 'Sector_8': 57, 'Sector_9': 39}


In [3]:

def build_design_matrix(records, date_stat, config):
    y_transform = config["y_transform"]
    x_transform = config["x_transform"]
    include_intercept = config.get("include_intercept", False)
    add_sector_dummies = config.get("sector_dummies", False)
    dummy_base = config.get("base_sector", BASE_SECTOR)

    y_values = []
    x_rows = []
    for rec in records:
        sec = rec["sector"]
        ret = rec["return"]
        if y_transform == "raw":
            y_val = ret
        elif y_transform == "sector_demean":
            y_val = ret - date_stat["sectors"][sec]["return"]["mean"]
        elif y_transform == "sector_zscore":
            std = date_stat["sectors"][sec]["return"]["std"]
            if std < 1e-12:
                y_val = 0.0
            else:
                y_val = (ret - date_stat["sectors"][sec]["return"]["mean"]) / std
        elif y_transform == "global_demean":
            y_val = ret - date_stat["overall"]["return"]["mean"]
        elif y_transform == "global_zscore":
            std = date_stat["overall"]["return"]["std"]
            if std < 1e-12:
                y_val = 0.0
            else:
                y_val = (ret - date_stat["overall"]["return"]["mean"]) / std
        else:
            raise ValueError(f"Unknown y_transform {y_transform}")
        y_values.append(y_val)

        row = []
        if include_intercept:
            row.append(1.0)
        for fname in FACTOR_NAMES:
            value = rec[fname]
            if x_transform == "raw":
                x_val = value
            elif x_transform == "sector_demean":
                x_val = value - date_stat["sectors"][sec]["factors"][fname]["mean"]
            elif x_transform == "sector_zscore":
                std = date_stat["sectors"][sec]["factors"][fname]["std"]
                if std < 1e-12:
                    x_val = 0.0
                else:
                    x_val = (value - date_stat["sectors"][sec]["factors"][fname]["mean"]) / std
            elif x_transform == "global_demean":
                x_val = value - date_stat["overall"]["factors"][fname]["mean"]
            elif x_transform == "global_zscore":
                std = date_stat["overall"]["factors"][fname]["std"]
                if std < 1e-12:
                    x_val = 0.0
                else:
                    x_val = (value - date_stat["overall"]["factors"][fname]["mean"]) / std
            else:
                raise ValueError(f"Unknown x_transform {x_transform}")
            row.append(x_val)
        if add_sector_dummies:
            for sec_name in SECTORS:
                if sec_name == dummy_base:
                    continue
                row.append(1.0 if rec["sector"] == sec_name else 0.0)
        x_rows.append(row)

    names = []
    if include_intercept:
        names.append("alpha")
    names.extend(FACTOR_NAMES)
    if add_sector_dummies:
        for sec_name in SECTORS:
            if sec_name == dummy_base:
                continue
            names.append(f"delta_{sec_name}")
    return y_values, x_rows, names


def xtx_matrix(x_rows):
    k = len(x_rows[0])
    matrix = [[0.0 for _ in range(k)] for _ in range(k)]
    for row in x_rows:
        for i in range(k):
            xi = row[i]
            for j in range(k):
                matrix[i][j] += xi * row[j]
    return matrix


def xty_vector(x_rows, y_values):
    k = len(x_rows[0])
    vector = [0.0 for _ in range(k)]
    for row, y in zip(x_rows, y_values):
        for i in range(k):
            vector[i] += row[i] * y
    return vector


def gaussian_solve(a_matrix, b_vector):
    n = len(a_matrix)
    aug = [a_matrix[i][:] + [b_vector[i]] for i in range(n)]
    for col in range(n):
        pivot_row = max(range(col, n), key=lambda r: abs(aug[r][col]))
        if abs(aug[pivot_row][col]) < 1e-12:
            raise ValueError("Singular matrix")
        if pivot_row != col:
            aug[col], aug[pivot_row] = aug[pivot_row], aug[col]
        pivot_val = aug[col][col]
        inv = 1.0 / pivot_val
        for j in range(col, n + 1):
            aug[col][j] *= inv
        for row in range(n):
            if row == col:
                continue
            factor = aug[row][col]
            if abs(factor) < 1e-18:
                continue
            for j in range(col, n + 1):
                aug[row][j] -= factor * aug[col][j]
    return [aug[i][n] for i in range(n)]


def invert_matrix(a_matrix):
    n = len(a_matrix)
    aug = [a_matrix[i][:] + [1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]
    for col in range(n):
        pivot_row = max(range(col, n), key=lambda r: abs(aug[r][col]))
        if abs(aug[pivot_row][col]) < 1e-12:
            raise ValueError("Singular matrix for inversion")
        if pivot_row != col:
            aug[col], aug[pivot_row] = aug[pivot_row], aug[col]
        pivot = aug[col][col]
        inv = 1.0 / pivot
        for j in range(col, 2 * n):
            aug[col][j] *= inv
        for row in range(n):
            if row == col:
                continue
            factor = aug[row][col]
            if abs(factor) < 1e-18:
                continue
            for j in range(col, 2 * n):
                aug[row][j] -= factor * aug[col][j]
    return [[aug[i][n + j] for j in range(n)] for i in range(n)]


def ols(y_values, x_rows):
    xtx = xtx_matrix(x_rows)
    xty = xty_vector(x_rows, y_values)
    beta = gaussian_solve([row[:] for row in xtx], xty[:])
    residuals = []
    for row, y in zip(x_rows, y_values):
        fitted = sum(row[i] * beta[i] for i in range(len(beta)))
        residuals.append(y - fitted)
    n = len(y_values)
    k = len(beta)
    rss = sum(res ** 2 for res in residuals)
    df_resid = n - k
    sigma2 = rss / df_resid if df_resid > 0 else 0.0
    xtx_inv = invert_matrix(xtx)
    std_err = [math.sqrt(sigma2 * xtx_inv[i][i]) for i in range(len(beta))]
    tstats = [beta[i] / std_err[i] if std_err[i] > 0 else float('inf') for i in range(len(beta))]
    return {
        "beta": beta,
        "std_err": std_err,
        "tstats": tstats,
        "residuals": residuals,
        "sigma2": sigma2,
        "df_resid": df_resid,
        "xtx_inv": xtx_inv,
    }


In [4]:

from collections import OrderedDict


def run_spec(data_by_date, date_stats, config):
    results = OrderedDict()
    for current_date, records in data_by_date.items():
        y_vals, x_rows, names = build_design_matrix(records, date_stats[current_date], config)
        ols_res = ols(y_vals, x_rows)
        results[current_date] = {
            "names": names,
            "beta": {name: ols_res["beta"][idx] for idx, name in enumerate(names)},
            "std_err": {name: ols_res["std_err"][idx] for idx, name in enumerate(names)},
            "tstats": {name: ols_res["tstats"][idx] for idx, name in enumerate(names)},
            "sigma2": ols_res["sigma2"],
            "df_resid": ols_res["df_resid"],
        }
    return results


def average_beta(results, names):
    acc = {name: 0.0 for name in names}
    count = 0
    for res in results.values():
        count += 1
        for name in names:
            acc[name] += res["beta"][name]
    return {name: acc[name] / count for name in names}


def compare_spec_betas(res1, res2, names, tolerance=1e-9):
    for date in res1.keys():
        for name in names:
            diff = abs(res1[date]["beta"][name] - res2[date]["beta"][name])
            if diff > tolerance:
                return False, (date, name, diff)
    return True, None


def verify_table_claims(data_by_date, date_stats, spec_results):
    output_lines = []
    metrics = {}

    ok_bd, info_bd = compare_spec_betas(spec_results['B'], spec_results['D'], FACTOR_NAMES)
    ok_cd, info_cd = compare_spec_betas(spec_results['C'], spec_results['D'], FACTOR_NAMES)
    metrics['B_vs_D_beta_equal'] = ok_bd
    metrics['C_vs_D_beta_equal'] = ok_cd
    output_lines.append(f"B vs D betas equal? {ok_bd} {'' if ok_bd else info_bd}")
    output_lines.append(f"C vs D betas equal? {ok_cd} {'' if ok_cd else info_cd}")

    max_diff_alpha_B = 0.0
    max_diff_delta_B = 0.0
    for date_key, res in spec_results['B'].items():
        stats = date_stats[date_key]
        beta_vec = [res['beta'][fname] for fname in FACTOR_NAMES]
        r0 = stats['sectors'][BASE_SECTOR]['return']['mean']
        x0 = [stats['sectors'][BASE_SECTOR]['factors'][fname]['mean'] for fname in FACTOR_NAMES]
        expected_alpha = r0 - sum(beta_vec[i] * x0[i] for i in range(len(FACTOR_NAMES)))
        diff_alpha = abs(expected_alpha - res['beta']['alpha'])
        max_diff_alpha_B = max(max_diff_alpha_B, diff_alpha)
        for sec in SECTORS:
            if sec == BASE_SECTOR:
                continue
            mean_r_s = stats['sectors'][sec]['return']['mean']
            mean_x_s = [stats['sectors'][sec]['factors'][fname]['mean'] for fname in FACTOR_NAMES]
            expected_delta = (mean_r_s - r0) - sum(beta_vec[i] * (mean_x_s[i] - x0[i]) for i in range(len(FACTOR_NAMES)))
            diff_delta = abs(expected_delta - res['beta'][f'delta_{sec}'])
            max_diff_delta_B = max(max_diff_delta_B, diff_delta)
    metrics['Spec_B_alpha_max_diff'] = max_diff_alpha_B
    metrics['Spec_B_delta_max_diff'] = max_diff_delta_B
    output_lines.append(f"Spec B alpha formula max diff: {max_diff_alpha_B:.3e}")
    output_lines.append(f"Spec B sector dummy formula max diff: {max_diff_delta_B:.3e}")

    max_diff_alpha_C = 0.0
    max_diff_delta_C = 0.0
    for date_key, res in spec_results['C'].items():
        stats = date_stats[date_key]
        r0 = stats['sectors'][BASE_SECTOR]['return']['mean']
        diff_alpha = abs(res['beta']['alpha'] - r0)
        max_diff_alpha_C = max(max_diff_alpha_C, diff_alpha)
        for sec in SECTORS:
            if sec == BASE_SECTOR:
                continue
            target = stats['sectors'][sec]['return']['mean'] - r0
            diff_delta = abs(res['beta'][f'delta_{sec}'] - target)
            max_diff_delta_C = max(max_diff_delta_C, diff_delta)
    metrics['Spec_C_alpha_max_diff'] = max_diff_alpha_C
    metrics['Spec_C_delta_max_diff'] = max_diff_delta_C
    output_lines.append(f"Spec C alpha matches sector mean within {max_diff_alpha_C:.3e}")
    output_lines.append(f"Spec C delta matches mean diff within {max_diff_delta_C:.3e}")

    intercept_like = [res['beta'].get('alpha', 0.0) for res in spec_results['D'].values() if 'alpha' in res['beta']]
    if intercept_like:
        max_alpha_D = max(abs(val) for val in intercept_like)
        metrics['Spec_D_alpha_max_abs'] = max_alpha_D
        output_lines.append(f"Spec D alpha max abs {max_alpha_D:.3e}")
    else:
        metrics['Spec_D_has_intercept'] = False
        output_lines.append("Spec D has no intercept term as expected")

    ok_e_c_alpha, _ = compare_spec_betas(spec_results['E'], spec_results['C'], ['alpha'] + [f'delta_{sec}' for sec in SECTORS if sec != BASE_SECTOR])
    metrics['Spec_E_vs_C_alpha_delta_equal'] = ok_e_c_alpha
    output_lines.append(f"Spec E intercept/dummies equal to C? {ok_e_c_alpha}")

    max_beta_diff_BE = 0.0
    max_tstat_diff_BE = 0.0
    for date_key in spec_results['B']:
        for fname in FACTOR_NAMES:
            diff_beta = abs(spec_results['B'][date_key]['beta'][fname] - spec_results['E'][date_key]['beta'][fname])
            diff_t = abs(spec_results['B'][date_key]['tstats'][fname] - spec_results['E'][date_key]['tstats'][fname])
            max_beta_diff_BE = max(max_beta_diff_BE, diff_beta)
            max_tstat_diff_BE = max(max_tstat_diff_BE, diff_t)
    metrics['Spec_E_vs_B_beta_max_diff'] = max_beta_diff_BE
    metrics['Spec_E_vs_B_tstat_max_diff'] = max_tstat_diff_BE
    output_lines.append(f"Spec E vs B beta max diff {max_beta_diff_BE:.3e}, t-stat max diff {max_tstat_diff_BE:.3e}")

    intercept_F = [res['beta'].get('alpha', 0.0) for res in spec_results['F'].values() if 'alpha' in res['beta']]
    if intercept_F:
        max_alpha_F = max(abs(val) for val in intercept_F)
        metrics['Spec_F_alpha_max_abs'] = max_alpha_F
        output_lines.append(f"Spec F intercept max abs {max_alpha_F:.3e}")
    else:
        metrics['Spec_F_has_intercept'] = False
        output_lines.append("Spec F has no intercept and relies on pure zscore structure")

    intercept_G = [res['beta'].get('alpha', 0.0) for res in spec_results['G'].values() if 'alpha' in res['beta']]
    if intercept_G:
        max_alpha_G = max(abs(val) for val in intercept_G)
        metrics['Spec_G_alpha_max_abs'] = max_alpha_G
        output_lines.append(f"Spec G intercept max abs {max_alpha_G:.3e}")
    ok_bg, info_bg = compare_spec_betas(spec_results['B'], spec_results['G'], FACTOR_NAMES)
    if ok_bg:
        metrics['Spec_G_betas_equal_B'] = True
        output_lines.append('Spec G betas equal to B (β matches but scaling issues remain)')
    else:
        metrics['Spec_G_betas_equal_B'] = False
        metrics['Spec_G_first_diff'] = info_bg
        diff_date, diff_name, diff_val = info_bg
        output_lines.append(f"Spec G differs from B (first diff at {diff_date} coef {diff_name} magnitude {diff_val:.3e})")
    max_tstat_gap_bg = 0.0
    for date_key in spec_results['B']:
        for fname in FACTOR_NAMES:
            diff = abs(spec_results['B'][date_key]['tstats'][fname] - spec_results['G'][date_key]['tstats'][fname])
            max_tstat_gap_bg = max(max_tstat_gap_bg, diff)
    metrics['Spec_G_vs_B_tstat_max_diff'] = max_tstat_gap_bg
    output_lines.append(f"Spec G vs B t-stats max diff {max_tstat_gap_bg:.3e}")

    tstat_diff = 0.0
    for date_key in spec_results['H']:
        for fname in FACTOR_NAMES:
            diff = abs(spec_results['A'][date_key]['tstats'][fname] - spec_results['H'][date_key]['tstats'][fname])
            tstat_diff = max(tstat_diff, diff)
    metrics['Spec_H_vs_A_tstat_max_diff'] = tstat_diff
    output_lines.append(f"Spec H vs A t-stats max diff {tstat_diff:.3e}")

    return ''.join(output_lines), metrics


In [5]:

specs = {
    'A': {"y_transform": "raw", "x_transform": "raw", "include_intercept": True, "sector_dummies": False},
    'B': {"y_transform": "raw", "x_transform": "raw", "include_intercept": True, "sector_dummies": True},
    'C': {"y_transform": "raw", "x_transform": "sector_demean", "include_intercept": True, "sector_dummies": True},
    'D': {"y_transform": "sector_demean", "x_transform": "sector_demean", "include_intercept": False, "sector_dummies": False},
    'E': {"y_transform": "raw", "x_transform": "sector_zscore", "include_intercept": True, "sector_dummies": True},
    'F': {"y_transform": "sector_zscore", "x_transform": "sector_zscore", "include_intercept": False, "sector_dummies": False},
    'G': {"y_transform": "raw", "x_transform": "sector_demean", "include_intercept": True, "sector_dummies": False},
    'H': {"y_transform": "global_zscore", "x_transform": "global_zscore", "include_intercept": True, "sector_dummies": False},
}

spec_results = {key: run_spec(data_by_date, date_stats, cfg) for key, cfg in specs.items()}
report, metrics = verify_table_claims(data_by_date, date_stats, spec_results)

first_date = next(iter(spec_results['A']))
print("Example spec A betas (first month):", spec_results['A'][first_date]['beta'])
print(report)
print("Key metrics:", metrics)

assert metrics['B_vs_D_beta_equal']
assert metrics['C_vs_D_beta_equal']
assert metrics['Spec_B_alpha_max_diff'] < 1e-12
assert metrics['Spec_B_delta_max_diff'] < 1e-12
assert metrics['Spec_C_alpha_max_diff'] < 1e-12
assert metrics['Spec_C_delta_max_diff'] < 1e-12
assert metrics['Spec_E_vs_B_beta_max_diff'] > 1e-3
assert metrics['Spec_E_vs_B_tstat_max_diff'] > 1.0
assert metrics['Spec_G_vs_B_tstat_max_diff'] > 1.0
assert metrics['Spec_H_vs_A_tstat_max_diff'] < 1e-10


Example spec A betas (first month): {'alpha': 0.0019553788537850668, 'x1': 0.04862690914293725, 'x2': -0.030613224935473384, 'x3': 0.022551705836291708, 'x4': 0.010330599329890012, 'x5': -0.0008971883195271425}
B vs D betas equal? True C vs D betas equal? True Spec B alpha formula max diff: 1.735e-16Spec B sector dummy formula max diff: 1.874e-16Spec C alpha matches sector mean within 1.665e-16Spec C delta matches mean diff within 1.717e-16Spec D has no intercept term as expectedSpec E intercept/dummies equal to C? TrueSpec E vs B beta max diff 1.677e-02, t-stat max diff 2.306e+00Spec F has no intercept and relies on pure zscore structureSpec G intercept max abs 2.200e-02Spec G betas equal to B (β matches but scaling issues remain)Spec G vs B t-stats max diff 1.604e+01Spec H vs A t-stats max diff 7.461e-14
Key metrics: {'B_vs_D_beta_equal': True, 'C_vs_D_beta_equal': True, 'Spec_B_alpha_max_diff': 1.734723475976807e-16, 'Spec_B_delta_max_diff': 1.8735013540549517e-16, 'Spec_C_alpha_max

In [49]:
spec_results['B'][date].keys()

dict_keys(['names', 'beta', 'std_err', 'tstats', 'sigma2', 'df_resid'])

In [52]:
for date, _ in date_stats.items():
    print(f"date {date}")
    print(spec_results['B'][date]['beta'])
    print(spec_results['C'][date]['beta'])
    print(spec_results['D'][date]['beta'])
    print(spec_results['G'][date]['beta'])

    # print(spec_results['B'][date]['tstats'])
    # print(spec_results['C'][date]['tstats'])
    # print(spec_results['D'][date]['tstats'])
    # print(spec_results['G'][date]['tstats'])

date 2018-01-31
{'alpha': -0.011549279911625496, 'x1': 0.04777677618293171, 'x2': -0.031443090370832885, 'x3': 0.02227049511242956, 'x4': 0.011038289504524595, 'x5': -0.0018043887819084144, 'delta_Sector_1': -0.007277470038082063, 'delta_Sector_2': 0.0015522226022976898, 'delta_Sector_3': 0.0040512620203834206, 'delta_Sector_4': 0.02948009201533236, 'delta_Sector_5': 0.03875274497394181, 'delta_Sector_6': 0.03776715672403555, 'delta_Sector_7': 0.022012551868313194, 'delta_Sector_8': -0.01245400457847317, 'delta_Sector_9': 0.03253278781245333}
{'alpha': -0.005868918766787643, 'x1': 0.04777677618293177, 'x2': -0.03144309037083292, 'x3': 0.02227049511242954, 'x4': 0.011038289504524627, 'x5': -0.001804388781908412, 'delta_Sector_1': -0.019008921549442503, 'delta_Sector_2': -0.010177389517523068, 'delta_Sector_3': 0.0024174931149757994, 'delta_Sector_4': 0.030706057043737543, 'delta_Sector_5': 0.023608378533294422, 'delta_Sector_6': 0.027272066093734997, 'delta_Sector_7': 0.0316460312766431

In [6]:

print("Average factor betas by specification (60 months):")
for spec_id in specs:
    averages = average_beta(spec_results[spec_id], FACTOR_NAMES)
    rounded = {k: round(v, 4) for k, v in averages.items()}
    print(spec_id, rounded)

sample_date = first_date
stats = date_stats[sample_date]
mean_r0 = stats['sectors'][BASE_SECTOR]['return']['mean']
mean_x0 = [stats['sectors'][BASE_SECTOR]['factors'][fname]['mean'] for fname in FACTOR_NAMES]
beta_B = spec_results['B'][sample_date]['beta']
alpha_formula = mean_r0 - sum(beta_B[fname] * mean_x0[idx] for idx, fname in enumerate(FACTOR_NAMES))
print("Spec B alpha vs formula on", sample_date, ":", beta_B['alpha'], alpha_formula)
print("Spec C alpha vs sector mean:", spec_results['C'][sample_date]['beta']['alpha'], mean_r0)
print("Spec E factor beta scaling relative to B (x1):", spec_results['E'][sample_date]['beta']['x1'], spec_results['B'][sample_date]['beta']['x1'])
print("Spec H factor t-stats vs A (x1):", spec_results['H'][sample_date]['tstats']['x1'], spec_results['A'][sample_date]['tstats']['x1'])


Average factor betas by specification (60 months):
A {'x1': 0.041, 'x2': -0.0287, 'x3': 0.0201, 'x4': -0.0005, 'x5': -0.0009}
B {'x1': 0.0409, 'x2': -0.0286, 'x3': 0.0202, 'x4': -0.0006, 'x5': -0.0009}
C {'x1': 0.0409, 'x2': -0.0286, 'x3': 0.0202, 'x4': -0.0006, 'x5': -0.0009}
D {'x1': 0.0409, 'x2': -0.0286, 'x3': 0.0202, 'x4': -0.0006, 'x5': -0.0009}
E {'x1': 0.0304, 'x2': -0.0198, 'x3': 0.014, 'x4': -0.0006, 'x5': -0.0009}
F {'x1': 0.662, 'x2': -0.4256, 'x3': 0.3021, 'x4': -0.0105, 'x5': -0.0232}
G {'x1': 0.0409, 'x2': -0.0286, 'x3': 0.0202, 'x4': -0.0006, 'x5': -0.0009}
H {'x1': 0.6347, 'x2': -0.4099, 'x3': 0.2881, 'x4': -0.0057, 'x5': -0.0173}
Spec B alpha vs formula on 2018-01-31 : -0.011549279911625496 -0.011549279911625496
Spec C alpha vs sector mean: -0.005868918766787643 -0.0058689187667876445
Spec E factor beta scaling relative to B (x1): 0.035214202132582695 0.04777677618293171
Spec H factor t-stats vs A (x1): 29.60032367150982 29.600323671509805
